<a href="https://colab.research.google.com/github/MuteebaAly/ML_Engineer_Internship/blob/main/Task_4/Headline_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets evaluate gradio accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer


Tokenization & Dataset:

In [ ]:
#Dataset aur Tokenizer load
dataset = load_dataset(r"fancyzhx/ag_news")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(10000))
test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(2000))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Fine-Tuning

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# Evaluation Metrics Setup
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

# Training Settings
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,               # TPU par thodi standard learning rate achhi rehti hai
    per_device_train_batch_size=32,   # TPU badi batch size (32 ya 64) ko makkhan ki tarah chalata hai
    num_train_epochs=1,               # 1 epoch bhi kafi hoga achhi accuracy ke liye
    weight_decay=0.01,
    logging_steps=100                 # Har 100 steps baad progress dikhaye
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Training Shuru!
trainer.train()

# Model Save karein
model.save_pretrained("./my_saved_model")
tokenizer.save_pretrained("./my_saved_model")
print("TPU Training Completed Successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.283041,0.291957,0.905000,0.905866


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TPU Training Completed Successfully!


In [ ]:
%%writefile app.py
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Page ki basic settings aur title
st.set_page_config(page_title="News Classifier", page_icon="📰", layout="centered")

st.title("📰 News Topic Classifier Using BERT")
st.write("Welcome Muteeba! Enter any news headline below, and our fine-tuned BERT model will predict its category in real-time.")

# 2. Model aur Tokenizer load karne ka function (cache lagaya hai takay baar baar load na ho)
@st.cache_resource
def load_my_model():
    try:
        model = AutoModelForSequenceClassification.from_pretrained("./my_saved_model")
        tokenizer = AutoTokenizer.from_pretrained("./my_saved_model")
    except:
        # Agar saved model na mile toh temporary internet se fresh load karein
        model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
        tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    return model, tokenizer

model, tokenizer = load_my_model()

# Labels ki mapping
labels_mapping = {0: "World News", 1: "Sports News", 2: "Business News", 3: "Sci/Tech News"}

# 3. User Input Box
headline = st.text_area("Enter News Headline Here:", placeholder="Type or paste a headline...")

# 4. Predict Button
if st.button("Predict Category", type="primary"):
    if headline.strip() == "":
        st.warning("Please write something first!")
    else:
        # Preprocessing & Tokenization
        inputs = tokenizer(headline, return_tensors="pt", truncation=True, max_length=128)

        model.eval()
        with torch.no_grad():
            outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

        # Prediction index nikalna
        prediction_index = torch.argmax(outputs.logits, dim=-1).item()
        final_result = labels_mapping[prediction_index]

        # Result ko khoobsurat box mein dikhana
        st.success(f"**Predicted Category:** {final_result}")

Writing app.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 41.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 2s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸⠙⠹⠸⠼⠴⠦your url is: https://blue-places-hope.loca.lt
